# 01 — LLM Behavior and Prompt Anatomy

## Scenario
Northstar’s classifier changes behavior after a request configuration change. This lab treats the request packet as an observable system and tests one variable at a time.

**Safety boundary:** this notebook uses Northstar's replay/live client. Replay mode is deterministic and offline; live mode is opt-in and uses the configured provider client. Its effects illustrate an experimental method of treating the prompt as a configurable packet.

In [ ]:
from pathlib import Path
from northstar.contracts import parse_structured
from northstar.runtime import get_client, estimate_tokens
from lab01 import CASES, EVIDENCE, INSTRUCTION, SupportClassification, build_requests

client = get_client(Path("fixtures/replays.json"))
REQUESTS = {request.case_id: request for request in build_requests()}

def show(case_id):
    request = REQUESTS[case_id]
    print(f"\n=== {case_id} ===")
    print("SYSTEM:\n" + (request.system or "(none)"))
    for message in request.messages:
        print(f"{message.role.upper()}:\n{message.text}")
    response = client.generate(request)
    print("RECORDED RESPONSE:\n" + response.text)
    parsed = response.parsed
    if parsed is None:
        parsed = parse_structured(SupportClassification, response.text).value
    print("PARSED VALUE:", parsed.model_dump())
    return request, response, parsed

## Baseline hypothesis

A precise classification instruction, approved evidence, and low variation (temperature 0) should classify clear cases while escalating the ambiguous payment case. We will score that baseline before changing anything.

In [ ]:
baseline_results = []
for case in CASES:
    _, response, parsed = show(f"b01/baseline/{case['id']}")
    baseline_results.append((case["id"], case["expected"], parsed.category))
print("Baseline results:", baseline_results)
assert sum(expected == observed for _, expected, observed in baseline_results) == 4

## Experiment 1 — position is a variable

Keep the cases and instruction fixed. Move the evidence into the synthetic middle position. This tells us to test source order on a real model, not to assume a universal effect.

In [ ]:
middle_results = []
for case in CASES:
    _, response, parsed = show(f"b01/middle/{case['id']}")
    middle_results.append((case["id"], case["expected"], parsed.category))
print("Middle-position results:", middle_results)
assert sum(expected == observed for _, expected, observed in middle_results) == 3

## Experiment 2 — sampling is a trade-off

Now keep evidence first but introduce a non-zero temperature. On a real model, run repeated samples and compare variation, task accuracy, and the cost of additional calls. Never infer correctness from lower temperature alone.

In [ ]:
_, _, temp_zero = show("b01/sampling/temp-0")
_, _, temp_nine = show("b01/sampling/temp-09")
assert temp_zero.category != temp_nine.category
padding = "The customer is a highly valued member. Please be polite.\n" * 10
print("Estimated padding tokens:", estimate_tokens(padding))
print("Estimated baseline message tokens:", estimate_tokens(EVIDENCE + "\nMessage: " + CASES[0]["message"]))
roles, _, _ = show("b01/structure/roles")
concatenated, _, _ = show("b01/structure/concatenated")
print("Role fingerprint:", roles.fingerprint())
print("Concatenated fingerprint:", concatenated.fingerprint())
assert roles.fingerprint() != concatenated.fingerprint()

## Failure injection — missing evidence

A refund decision without approved evidence must not become a confident refund label. This is a context/contract failure, not a request for stronger role wording. The safe result is `unknown`, followed by clarification or escalation in the surrounding application.

In [ ]:
_, _, weak = show("b01/missing/weak")
print("In the recorded run, the weak instruction produced:", weak.category)
assert weak.category == "refund"

## The Fix: Strong Instruction and Abstention

We must instruct the model to abstain if the evidence does not support a clear category.

In [ ]:
_, _, abstain = show("b01/missing/abstain")
print("In the recorded run, the explicit fallback produced:", abstain.category)
assert abstain.category == "unknown"

## Takeaway

The assertions above describe the deterministic recorded run. Change one prompt variable, rerun the lab, and measure the trade-off.

## References

See the course README for the folded reference material and links.

## Reading the recorded experiment

The baseline is deliberately small: the task, approved policy text, four messages,
and a typed response schema are fixed before any comparison is made. The clear
refund, shipping, and account cases test ordinary routing. The duplicate-charge
case is a boundary case: the policy text does not say that every charge anomaly is
a refund, so `unknown` is the safer expected label. This is the important habit
from the original lesson: write the expected behavior before looking at outputs.

The middle-position run changes only where the evidence appears. The repeated
politeness text is synthetic padding, not useful context. It lets us ask whether
the same policy is used as reliably when it is surrounded by irrelevant material.
The recorded result is not a universal claim about every model or context window;
it is one measured point in a position-and-density experiment. In a production
evaluation, repeat this with the actual model, prompt length, retrieval order, and
task distribution.

The sampling comparison changes only temperature. A difference between the two
recorded responses is evidence that the ambiguous boundary is sensitive to
sampling, not evidence that a higher temperature is intrinsically better. Track
accuracy, abstention quality, token use, latency, and cost together. A prompt
change should be treated like a code change: freeze the cases, preserve the
request fingerprint, and inspect the exact response before drawing a conclusion.

Finally, the missing-evidence experiment demonstrates why a schema alone is not a
grounding policy. The weak instruction permits a confident category in the
recorded run. The explicit fallback makes absence of evidence an allowed,
testable output. Role-separated system and user messages also produce a different
fingerprint from concatenated text, so the message structure is part of the
request contract and should be versioned and measured.

Prompt anatomy also includes the boundary between instructions and data. A
delimiter can make that boundary easier to inspect, but it cannot authorize a
claim or replace an application check. In this lesson, the approved policy text
is visible in every request, the customer message is labelled separately, and
the response schema limits the category vocabulary. Those are independent
controls, so a learner can change one at a time and see which metric moves.
Record the request fingerprint, selected temperature, estimated tokens, and
observed parsed value together; otherwise a later comparison may accidentally
mix prompt changes with sampling or fixture changes.